# 02. Regime Relativity

This notebook computes within-regime effect sizes (`G2-G1` and `G4-G3`) across layers and visualises the sign reversals that support the Regime-Relativity Principle.
        


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
DATA = ROOT / 'data'
FIGURES = ROOT / 'figures'
FIGURES.mkdir(exist_ok=True)
sys.path.insert(0, str(ROOT))

from src.statistical_analysis import cohens_d, two_way_anova, stratified_logistic_auc

sns.set_theme(style='whitegrid', font_scale=1.1)
pd.set_option('display.max_columns', 120)

metrics = [
    'speed', 'radius_of_gyration', 'effective_dim', 'dir_consistency', 'tortuosity',
    'msd_exponent', 'cos_to_late_window', 'cos_to_running_mean', 'stabilization', 'dist_slope'
]
labels = {
    'speed': 'Speed',
    'radius_of_gyration': 'Radius of gyration',
    'effective_dim': 'Effective dimension',
    'dir_consistency': 'Directional consistency',
    'tortuosity': 'Tortuosity',
    'msd_exponent': 'MSD exponent',
    'cos_to_late_window': 'Cosine to late window',
    'cos_to_running_mean': 'Cosine to running mean',
    'stabilization': 'Stabilization rate',
    'dist_slope': 'Distance slope',
}
df = pd.read_csv(DATA / 'qwen05b' / 'unified_metrics.csv')
        


In [ ]:
rows = []
for metric in metrics:
    for layer in sorted(df['layer'].unique()):
        layer_df = df[df['layer'] == layer]
        g2 = layer_df[layer_df['group'] == 'G2'][metric]
        g1 = layer_df[layer_df['group'] == 'G1'][metric]
        g4 = layer_df[layer_df['group'] == 'G4'][metric]
        g3 = layer_df[layer_df['group'] == 'G3'][metric]
        rows.append({'metric': metric, 'layer': layer, 'comparison': 'Direct', 'cohen_d': cohens_d(g2, g1)})
        rows.append({'metric': metric, 'layer': layer, 'comparison': 'CoT', 'cohen_d': cohens_d(g4, g3)})

effects = pd.DataFrame(rows)
direct = effects[effects['comparison'] == 'Direct'].pivot(index='metric', columns='layer', values='cohen_d').loc[metrics]
cot = effects[effects['comparison'] == 'CoT'].pivot(index='metric', columns='layer', values='cohen_d').loc[metrics]
direct.index = [labels[m] for m in direct.index]
cot.index = [labels[m] for m in cot.index]

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)
sns.heatmap(direct, cmap='RdBu_r', center=0, vmin=-3, vmax=3, ax=axes[0], cbar=False)
sns.heatmap(cot, cmap='RdBu_r', center=0, vmin=-3, vmax=3, ax=axes[1], cbar_kws={'label': "Cohen's d"})
axes[0].set_title('Direct regime (G2 - G1)')
axes[1].set_title('CoT regime (G4 - G3)')
for ax in axes:
    ax.set_xlabel('Layer')
axes[0].set_ylabel('Metric')
plt.tight_layout()
plt.savefig(FIGURES / 'repro_02_regime_relativity.png', dpi=300)
plt.show()

peak_summary = []
for metric in metrics:
    sub = effects[effects['metric'] == metric]
    direct_peak = sub[sub['comparison'] == 'Direct'].iloc[sub[sub['comparison'] == 'Direct']['cohen_d'].abs().argmax()]
    cot_peak = sub[sub['comparison'] == 'CoT'].iloc[sub[sub['comparison'] == 'CoT']['cohen_d'].abs().argmax()]
    peak_summary.append({
        'metric': metric,
        'direct_peak_d': float(direct_peak['cohen_d']),
        'cot_peak_d': float(cot_peak['cohen_d']),
        'sign_flip': np.sign(direct_peak['cohen_d']) != np.sign(cot_peak['cohen_d']),
    })
peak_summary = pd.DataFrame(peak_summary)
peak_summary
        
